# Módulo Geoespacial — Capa técnica transversal

**Bloque C — Técnicas transversales** | Módulo: `geoespacial`

---

## ¿Qué es este módulo y por qué existe?

El módulo geoespacial **no es una línea temática ambiental en sí misma**. Es la capa técnica transversal que da soporte espacial a todas las demás líneas del repositorio.

Cualquier análisis ambiental en Colombia tiene una dimensión geográfica inevitable:
- ¿Dónde están las estaciones de monitoreo? (sistemas de información)
- ¿Qué tan lejos está un páramo de una zona urbana? (páramos, ordenamiento territorial)
- ¿Dónde se agrupan los episodios de contaminación? (calidad del aire)
- ¿Cuál es la cuenca que aporta más caudal? (oferta hídrica, POMCA)
- ¿Qué municipios tienen mayor riesgo por inundación? (gestión del riesgo)

**Sin análisis espacial, muchos análisis ambientales colombianos son incompletos.**

---

## Sistema de coordenadas en Colombia — Decisión crítica antes de cualquier análisis

Colombia tiene cuatro sistemas de referencia relevantes:

| Sistema | EPSG | Cuándo usar |
|---|---|---|
| **MAGNA-SIRGAS** | 4686 | Coordenadas geográficas (lat/lon). Sistema oficial de Colombia. Compatible con GPS/GNSS. |
| **CTM12 (Origen Nacional)** | 9377 | Coordenadas proyectadas en metros. Estándar desde Res. IGAC 370/2021. Usar para cálculos de área y distancia. |
| **WGS84** | 4326 | Interoperabilidad con plataformas internacionales (Google Maps, QGIS base layers). |
| **Pseudo-Mercator Web** | 3857 | Solo para visualización web. **Nunca para cálculos de área o distancia.** |

**Error frecuente:** calcular áreas en WGS84 (grados decimales). Siempre reproyectar a CTM12 (metros) antes de calcular áreas o distancias.

```python
import geopandas as gpd

gdf = gpd.read_file("mi_capa.shp")          # crs original (puede ser 4326 o 4686)
gdf_ctm12 = gdf.to_crs("EPSG:9377")        # reproyectar a CTM12 para cálculos
area_km2 = gdf_ctm12.geometry.area / 1e6   # área en km²
```

---

## Líneas temáticas que alimenta este módulo

| Línea | Uso espacial principal |
|---|---|
| Áreas protegidas, páramos, humedales | Delimitación de polígonos, cálculo de área, solapamiento con usos del suelo |
| Rondas hídricas, POMCA | Extracción de redes de drenaje (D8), zonificación hidrográfica |
| Gestión del riesgo | Mapas de amenaza, análisis de vulnerabilidad espacial |
| Ordenamiento territorial | Cartografía POT, análisis de conflictos de uso |
| Calidad del aire | Interpolación de concentraciones, mapas de exposición poblacional |
| Sistemas de información | Densidad de estaciones, análisis de cobertura territorial |
| Predios para conservación | Georreferenciación, cruce con RUNAP y ecosistemas |

---

## Contenidos de este notebook

1. Revisión de sistemas de coordenadas y proyecciones Colombia
2. Interpolación espacial con IDW — caso de aplicación a estaciones de monitoreo
3. Autocorrelación espacial (I de Moran) — detección de hotspots ambientales
4. Cuándo usar análisis espacial vs. análisis temporal
5. Cómo conectar con las demás líneas del repositorio

> **Documentación de dominio completa:** [`docs/fuentes/geoespacial.md`](../../docs/fuentes/geoespacial.md)

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "geoespacial"
VARIABLE = "valor"
UNIDAD = ""

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `geoespacial` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "geoespacial.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Tipos de datos geoespaciales en el contexto ambiental colombiano

### Capas que maneja este módulo

**Datos raster (cuadrículas):**
- Imágenes satelitales multiespectrales: Landsat 8/9 (NASA, 30m), Sentinel-2 (ESA, 10m)
- Modelos Digitales de Elevación (DEM/SRTM): fundamental para hidrología y riesgo
- Humedad del suelo: SMAP L4 (NASA)
- Coberturas de la tierra: CORINE Land Cover para Colombia (IDEAM/IGAC)

**Datos vectoriales (puntos, líneas, polígonos):**
- Estaciones hidrometeorológicas georreferenciadas (DHIME/IDEAM)
- Límites administrativos: municipios, departamentos, jurisdicciones CAR (DANE/IGAC)
- Polígonos de áreas protegidas (RUNAP/MADS)
- Cuencas hidrográficas y zonificación (IDEAM)
- Rondas hídricas, humedales, páramos (MADS)

**Datos de red:**
- Redes de drenaje (extraídas de DEM con WhiteboxTools)
- Vías (para análisis de accesibilidad a servicios o estaciones)

### Variables de ejemplo en este notebook

El notebook usa datos sintéticos de concentración de PM2.5 en estaciones distribuidas por Colombia — un caso de uso típico de interpolación espacial para mapas de exposición.

**Estructura de datos esperada:**
```
estacion_id | lat    | lon     | pm25_ugm3 | fecha
------------|--------|---------|-----------|----------
IDEAM_001   | 4.6534 | -74.083 | 18.4      | 2024-01-15
IDEAM_002   | 6.2442 | -75.574 | 32.1      | 2024-01-15
...
```

> Colocar el archivo de estaciones en `data/raw/estaciones_georreferenciadas.csv`.

In [ ]:
# df = load_csv("data/raw/geoespacial.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "valor": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 2. Validación y EDA espacial

### ¿Qué validar antes de cualquier análisis geoespacial?

La validación de datos espaciales tiene dimensiones adicionales a la validación tabular:

**1. Sistema de referencia (CRS):**
```python
import geopandas as gpd
gdf = gpd.read_file("estaciones.shp")
print(gdf.crs)  # ¿Cuál CRS? ¿Es el correcto?
# Si es None: datos sin CRS definido — peligroso, consultar metadatos de la fuente
```

**2. Extensión geográfica (bounding box):**
- Colombia aproximado: lat [-4.2, 12.6], lon [-81.7, -66.8]
- Puntos fuera de este rango: errores de digitación en coordenadas.

**3. Geometrías inválidas:**
```python
gdf[~gdf.is_valid]  # polígonos con auto-intersecciones, etc.
gdf["geometry"] = gdf["geometry"].make_valid()  # corregir
```

**4. Distribución espacial:**
- Mapa de puntos de estaciones: ¿hay concentración en zona andina?
- Índice de dispersión espacial o densidad de Kernel para diagnosticar el sesgo de cobertura.

**5. Metadatos de las capas:**
- ¿Cuándo fue la última actualización del shapefile?
- ¿Escala de captura? (1:100.000 vs 1:25.000 — diferencia crítica para análisis de detalle)

> La validación espacial es previa a `validate()` tabular. Los errores de CRS o geometría inválida pueden generar resultados silenciosamente incorrectos.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_geoespacial.html",
       title="EDA — Geoespacial", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Interpolación espacial — IDW vs. Kriging

### ¿Cuándo interpolar y por qué?

Las estaciones de monitoreo son puntos discretos. Para generar un mapa continuo de concentraciones (ej. PM2.5 en todo Colombia) se necesita **interpolación espacial**: estimar el valor en ubicaciones sin estación a partir de las estaciones cercanas.

### IDW — Distancia Inversa Ponderada

**Principio:** los puntos más cercanos tienen más influencia, inversamente proporcional a su distancia.

```
V(x) = Σ(Vi / di^p) / Σ(1 / di^p)
```
donde `di` = distancia al punto i, `p` = potencia (usualmente 2).

**Cuándo usar IDW:**
- Análisis exploratorios y visualizaciones rápidas.
- Cuando no se tiene tiempo de ajustar un variograma (Kriging).
- Datos con distribución espacial relativamente uniforme.

**Limitaciones de IDW:**
- No modela la autocorrelación espacial explícitamente.
- El parámetro `p` es arbitrario.
- Genera "bulls-eyes" (círculos alrededor de cada estación) — artificiales.

### Kriging Ordinario

**Principio:** Mejor Estimador Lineal Insesgado (BLUE) que modela la autocorrelación espacial mediante un variograma experimental ajustado a un modelo teórico (esférico, exponencial, gaussiano).

**Cuándo usar Kriging:**
- Estudios de suelos, agua subterránea, distribución de contaminantes.
- Cuando se requiere cuantificar la incertidumbre de la estimación (varianza de Kriging).
- La ficha de dominio indica que Kriging supera a IDW para propiedades físicas del suelo en Colombia.

**Flujo de Kriging con pykrige:**
```python
from pykrige.ok import OrdinaryKriging

ok = OrdinaryKriging(
    x=gdf["lon"], y=gdf["lat"], z=gdf["pm25_ugm3"],
    variogram_model="spherical",
    verbose=False
)
z_interp, ss_interp = ok.execute("grid", grid_lon, grid_lat)
```

> **Recomendación para esta línea:** usar IDW para exploración y Kriging para productos finales de calidad.

In [ ]:
plot_series(df, "fecha", "valor", title="Geoespacial — valor ()")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "valor", period="month")
plt.show()

## 4. Estadística espacial — I de Moran y detección de hotspots

### ¿Qué es la autocorrelación espacial y por qué importa?

La **Primera Ley de Tobler:** "Todo está relacionado con todo lo demás, pero las cosas cercanas están más relacionadas que las distantes."

En términos estadísticos: si hay autocorrelación espacial positiva, los valores similares se agrupan en el espacio. Esto viola el supuesto de independencia de las observaciones que asumen la mayoría de los modelos estadísticos clásicos.

**Consecuencia:** si se aplica regresión lineal o ANOVA sin verificar autocorrelación espacial, los errores estándar estarán subestimados y las pruebas de hipótesis serán inválidas.

### I de Moran Global

Mide si el fenómeno está **agrupado** (I > 0), **disperso** (I < 0) o **aleatorio** (I ≈ 0) en el espacio.

| Resultado | Interpretación ambiental |
|---|---|
| I > 0, p < 0.05 | Los episodios de alta contaminación se agrupan geográficamente — hay hotspots |
| I ≈ 0, p > 0.05 | La contaminación está distribuida aleatoriamente — no hay patrón espacial |
| I < 0, p < 0.05 | Los valores altos y bajos se alternan en el espacio — dispersión (raro en contaminación) |

### LISA (Local Indicators of Spatial Association) — Hotspots

El I de Moran local identifica clústeres específicos:
- **High-High (HH):** zona de alta concentración rodeada de alta concentración — hotspot real.
- **Low-Low (LL):** zona de baja concentración rodeada de baja — coldspot.
- **High-Low / Low-High:** valores atípicos espaciales — investigar causas.

**Aplicación ambiental:**
- Calidad del aire: identificar municipios con alta contaminación que "contagian" a vecinos.
- Deforestación: detectar frentes de deforestación activos (SMByC/IDEAM).
- Gestión del riesgo: zonas de acumulación de riesgo hídrico.

```python
from esda.moran import Moran, Moran_Local
from libpysal.weights import Queen

w = Queen.from_dataframe(gdf_municipios)
w.transform = "R"  # row-standardize
mi = Moran(gdf_municipios["pm25_media"], w)
print(f"I de Moran: {mi.I:.3f} | p-valor: {mi.p_sim:.4f}")
```

In [ ]:
summarize(df)

## 5. Cuándo usar análisis espacial vs. análisis temporal

### Árbol de decisión metodológico

Esta es una de las preguntas más importantes al comenzar un análisis ambiental:

```
¿Cuál es la pregunta de investigación?
│
├── "¿Cómo ha cambiado [variable] a lo largo del tiempo en [lugar]?"
│   → Análisis TEMPORAL: series de tiempo, ARIMA, tendencias
│   → Ejemplos: evolución de caudal en la estación Puente Lleras, tendencia de PM2.5 en Bogotá
│
├── "¿Cómo se distribuye [variable] en el espacio en [momento]?"
│   → Análisis ESPACIAL: interpolación, autocorrelación, mapas
│   → Ejemplos: mapa de concentración de PM2.5 en Colombia en enero 2024
│
├── "¿Hay episodios en [lugar] y [tiempo] donde [variable] supera umbrales?"
│   → ESPACIO-TEMPORAL: ambos juntos
│   → Ejemplos: episodios de contaminación en valle del Cauca durante inversión térmica
│
└── "¿Qué factores explican la distribución de [variable] en el territorio?"
    → REGRESIÓN ESPACIAL: GWR, SAR, SEM
    → Ejemplos: ¿qué variables socioecómicas explican la deforestación por municipio?
```

### Casos de uso típicos en el repositorio

| Situación | Análisis recomendado | Módulo |
|---|---|---|
| Episodio de contaminación (calidad del aire) | Espacial: IDW/Kriging para mapear extensión | geoespacial + calidad_aire |
| Distribución de estaciones IDEAM | Espacial: densidad de Kernel, análisis de cobertura | geoespacial + sistemas_informacion |
| Evolución del caudal en una estación | Temporal: ARIMA, Mann-Kendall | oferta_hidrica |
| Expansión de deforestación 2000-2024 | Espacio-temporal: multitemporal + hotspots | geoespacial + cambio_climatico |
| Validación cruzada espacial de modelos | Espacial: bloqueo espacial en k-fold | geoespacial (cualquier línea) |

### Validación cruzada espacial — concepto crítico

En datos espaciales, la validación cruzada aleatoria (k-fold estándar) es incorrecta porque los pliegues de entrenamiento y prueba comparten autocorrelación espacial — el modelo "hace trampa".

**Solución:** usar bloqueo espacial (spatial block cross-validation):
```python
# Dividir el espacio en bloques geográficos (ej. por departamento o cuadrícula)
# Los bloques completos van a train o test — nunca divididos
```

> Documentar en `docs/decisiones.md` si se usa validación cruzada espacial en lugar de temporal para alguna línea.

In [ ]:
ts = df.set_index("fecha")["valor"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Normas colombianas con dimensión espacial

### ¿Dónde aplican las normas de calidad en el espacio?

Algunas normas colombianas tienen umbrales que aplican de forma diferenciada según la ubicación geográfica o el uso del suelo:

**Calidad del aire (Res. 2254/2017 + config.NORMA_CO):**
- Los umbrales son nacionales, pero la exposición poblacional es heterogénea.
- Un mapa de excedencias combina la concentración interpolada con la densidad poblacional DANE.
- Municipios con alta concentración Y alta densidad poblacional son prioridad de gestión.

**Calidad del agua (Res. 2115/2007 + config.NORMA_AGUA_POTABLE):**
- Los umbrales aplican por punto de captación o de vertimiento — no por área.
- El análisis espacial permite identificar cuencas con alta frecuencia de incumplimiento.

**Rondas hídricas y áreas protegidas:**
- Las normas definen distancias mínimas (metros desde cuerpos de agua).
- El análisis espacial es la única forma de verificar cumplimiento — requiere buffers geométricos.

```python
# Ejemplo: verificar si una infraestructura respeta ronda hídrica de 30m
import geopandas as gpd

cuerpos_agua = gpd.read_file("data/raw/red_hidrica.shp").to_crs("EPSG:9377")
infraestructura = gpd.read_file("data/raw/infraestructura.shp").to_crs("EPSG:9377")

ronda = cuerpos_agua.buffer(30)  # 30 metros en CTM12
conflictos = infraestructura[infraestructura.intersects(ronda.union_all())]
print(f"Infraestructuras en conflicto con ronda hídrica: {len(conflictos)}")
```

In [ ]:
rep = exceedance_report(df["valor"], variable="valor")
if rep.empty:
    print("Sin normas colombianas registradas para 'valor'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Problemas frecuentes y mitigaciones en datos espaciales colombianos

### Proyecciones mixtas — el problema más común

Colombia tiene un historial de múltiples orígenes cartográficos locales (Bogotá, Oeste, Este, Cali). Antes de la Resolución IGAC 370/2021 (CTM12), cada entidad usaba su propio origen.

**Diagnóstico:**
```python
# Si al superponer dos capas no coinciden visualmente:
print(gdf1.crs, gdf2.crs)  # ¿Son iguales?
# Si crs es None: la capa no tiene CRS definido — peligroso
```

**Mitigación:**
```python
# Reproyectar siempre al sistema de referencia de trabajo
gdf1 = gdf1.to_crs("EPSG:9377")
gdf2 = gdf2.to_crs("EPSG:9377")
```

### Estaciones dispersas y vacíos de datos — sesgo de cobertura

Las estaciones del IDEAM están concentradas en la zona andina. La interpolación en Amazonia/Orinoquía produce resultados con alta incertidumbre.

**ADR-002 aplicado al dominio espacial:** no eliminar estaciones con valores extremos antes de analizar — un valor alto de PM2.5 en una estación aislada puede ser la única señal de un episodio real.

**Estrategias:**
- Kriging con varianza de estimación: muestra la incertidumbre como mapa de error.
- MissForest: imputación espacial para completar rasters con datos faltantes.
- Documentar explícitamente qué regiones tienen baja cobertura de monitoreo.

### Resoluciones mixtas y nubosidad en imágenes satelitales

En los Andes y Amazonía, la nubosidad permanente limita las imágenes ópticas (Sentinel-2, Landsat).

**Solución:**
- Combinar con datos de radar (Sentinel-1, SAR): penetra nubes.
- Compositar imágenes de múltiples fechas tomando el percentil 10 (mediana de valores claros).

In [ ]:
df_clean = impute(df.copy(), cols=["valor"], method="linear")
print(f"Faltantes antes: {df["valor"].isna().sum()} | "
      f"después: {df_clean["valor"].isna().sum()}")

## 7. Integración con el ciclo estadístico del repositorio

### Cómo este módulo alimenta las otras líneas

El módulo geoespacial no tiene un flujo predictivo propio (los modelos de ML/ARIMA se ejecutan en cada línea temática). Su rol es **enriquecer** el análisis de las otras líneas con información espacial:

**EDA espacial (antes de cualquier modelado):**
```python
# ¿Las series temporales de diferentes estaciones son comparables?
# Verificar que las estaciones pertenezcan a la misma unidad homogénea
# (mismo piso térmico, misma cuenca, misma zona climática)
```

**Variables espaciales como covariables en modelos predictivos:**
```python
# Para SARIMAX en calidad del aire (ADR en CLAUDE.md):
# Las variables meteorológicas de estaciones cercanas se obtienen por
# interpolación espacial IDW antes de usarlas como exógenas en SARIMAX

# Para modelos de oferta hídrica:
# El área de la cuenca aportante (calculada de DEM) es covariable fundamental
```

**Validación cruzada espacial en lugar de temporal:**
```python
# Para datos con alta autocorrelación espacial:
# Usar bloques espaciales (departamento, cuenca) para train/test
# En lugar de bloques temporales estándar del walk_forward()
```

**Mapas de resultados (outputs):**
```python
# El resultado de cualquier análisis puede representarse cartográficamente:
import folium

m = folium.Map(location=[4.5, -74.0], zoom_start=6)
folium.GeoJson(gdf_resultados, tooltip=folium.GeoJsonTooltip(["municipio", "pm25_pred"])).add_to(m)
m.save("data/output/reports/mapa_resultados.html")
```

### Librería Python recomendada para cada tarea

| Tarea | Librería principal | Alternativa |
|---|---|---|
| Lectura/escritura vectorial | `geopandas` | `fiona` |
| Lectura raster | `rasterio` / `rioxarray` | `xarray` + `netCDF4` |
| Kriging / variograma | `pykrige` | R: `gstat` via rpy2 |
| Autocorrelación espacial | `esda` + `libpysal` | `pysal` |
| Red de drenaje (hidrología) | `WhiteboxTools` | `pysheds` |
| Visualización interactiva | `folium` | `contextily` + matplotlib |
| Reproyección | `pyproj` | integrado en `geopandas` |

In [ ]:
ts = df_clean.set_index("fecha")["valor"]

models = {
    # "Kriging":   ver spatial/interpolation.py para datos espaciales
    "RandomForest": get_model("random_forest", lags=[1,2,3,6,12]),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 8. Síntesis — Cuándo y cómo usar este módulo

### Checklist antes de comenzar cualquier análisis con dimensión espacial

- [ ] ¿Tengo el CRS documentado para cada capa? (EPSG:4686, 9377, o 4326)
- [ ] ¿Reproyecté a CTM12 (EPSG:9377) para cálculos de área/distancia?
- [ ] ¿Verifiqué geometrías válidas (`gdf.is_valid.all()`)?
- [ ] ¿Tengo el shapefile de Colombia actualizado para delimitación?
- [ ] ¿Documenté la escala de captura de las capas base (1:100.000 o 1:25.000)?
- [ ] ¿Apliqué I de Moran antes de usar regresión con datos espaciales?

### Decisiones metodológicas de este módulo

- **IDW para exploración, Kriging para producción:** IDW es más rápido; Kriging es más preciso y cuantifica la incertidumbre.
- **ADR-002 aplicado:** no eliminar estaciones atípicas antes de interpolar — pueden ser señales reales.
- **Validación cruzada espacial:** usar bloqueo por unidad geográfica, no k-fold aleatorio.
- **Módulo `spatial/` pendiente:** ver `docs/fuentes/geoespacial.md` — se planea crear en fase futura.

---

## 9. Cómo usar este módulo con datos reales

### Fuente 1: Shapefile de Colombia (límites administrativos)

```python
# Opción A: datos.gov.co (DANE/IGAC)
# https://www.datos.gov.co → buscar "Marco Geoestadístico Nacional"
# Descargar MGN_MPIO_POLITICO.shp (municipios) o MGN_DPTO_POLITICO.shp (departamentos)

import geopandas as gpd
municipios = gpd.read_file("data/raw/MGN_MPIO_POLITICO.shp")
print(municipios.crs)  # Verificar que sea EPSG:4686 o EPSG:9377

# Opción B: descarga directa con requests (si el portal tiene API)
# https://geoportal.igac.gov.co
```

### Fuente 2: Estaciones IDEAM georreferenciadas

```python
# DHIME — Sistema de Información del Recurso Hídrico
# http://dhime.ideam.gov.co → Catálogo de estaciones → Exportar CSV

estaciones = pd.read_csv("data/raw/estaciones_ideam.csv")
gdf_estaciones = gpd.GeoDataFrame(
    estaciones,
    geometry=gpd.points_from_xy(estaciones["longitud"], estaciones["latitud"]),
    crs="EPSG:4686"  # MAGNA-SIRGAS
)

# Reproyectar para análisis
gdf_estaciones_ctm12 = gdf_estaciones.to_crs("EPSG:9377")
```

### Fuente 3: Imágenes satelitales (Sentinel-2 o Landsat)

```python
# Copernicus Open Access Hub: https://scihub.copernicus.eu
# NASA Earth Explorer: https://earthexplorer.usgs.gov
# Google Earth Engine (requiere cuenta): https://earthengine.google.com

import rasterio
with rasterio.open("data/raw/S2_20240115_B04.tif") as src:
    banda_roja = src.read(1)
    meta = src.meta
    print(f"CRS: {src.crs} | Resolución: {src.res} m")
```

### Fuente 4: Capas IGAC especializadas

```python
# GeoPortal IGAC: https://geoportal.igac.gov.co
# Disponible: cartografía básica, suelos, catastro, red geodésica

# Capas relevantes para el repositorio:
# - Unidades de suelo (para calidad de suelos, agricultura)
# - Zonas de vida Holdridge (para páramos, oferta hídrica)
# - Red hídrica (para rondas hídricas, POMCA)
# - Predios rurales (para predios de conservación)
```

### Integración con las normas colombianas del repositorio

```python
# Para mapas de excedencias de calidad del aire:
from estadistica_ambiental.config import NORMA_CO

umbral_pm25 = NORMA_CO["pm25"]["anual"]  # 25 µg/m³ (Res. 2254/2017)

# Mapa de municipios que exceden el umbral:
gdf_municipios["excede_norma"] = gdf_municipios["pm25_media_anual"] > umbral_pm25
gdf_municipios.plot(column="excede_norma", cmap="RdYlGn_r", legend=True)
```